[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/planessoria-ui/ModelVinya/blob/main/auto_labeling/safata_opencv.ipynb)

# 🍇 ModelVinya · Safata — mesura de baies (OpenCV, sense SAM 3)

Detecta **totes les baies** soltes sobre fons blanc (amb regle) i genera un projecte `.json` de ModelVinya.
Fa servir **només OpenCV** (color verd + watershed): **no cal GPU, ni token, ni SAM 3, ni instal·lar res**.
Després obres el `.json` a l'anotador, hi poses l'escala amb el regle i exportes.

> Ordre: executa les cel·les de dalt a baix. Comença amb `CALIBRA_1 = True` per provar 1 foto; quan surti bé, posa-ho a `False` (o fes servir la cel·la de lots).

In [ ]:
# 1) Carregar imatges (Drive) + funcions auxiliars  (sense instal·lar res; Colab ja porta cv2/numpy/PIL)
import os, glob, io, json, base64
import numpy as np, cv2
from PIL import Image, ImageOps
from google.colab import drive
drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/Mydrive racimos'   # ← la teva carpeta de fotos de safata
exts = ('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG')
image_paths = sorted(p for e in exts for p in glob.glob(os.path.join(CARPETA, e)))
print(len(image_paths), 'imatges trobades'); print(image_paths[:5])

def open_img(path):
    return ImageOps.exif_transpose(Image.open(path)).convert('RGB')   # respecta la rotacio EXIF

def data_url(img):
    buf = io.BytesIO(); img.convert('RGB').save(buf, format='JPEG', quality=92)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')

EMPTY_FICHA  = {"id_racimo": "", "fecha": "", "variedad": "", "fase_fenologica": "",
                "tratamiento": "", "vigor": "", "orientacion": "", "sistema_conduccion": ""}
EMPTY_VERDAD = {"N_total_real": "", "Diam_real_mm": "", "Peso_baya_real_g": "", "Peso_racimo_real_g": ""}

def save_project(images_out, name='modelvinya_safata.json'):
    proj = {"app": "ModelVinya", "version": 1,
            "savedAt": __import__('datetime').datetime.utcnow().isoformat(), "images": images_out}
    with open(name, 'w') as f: json.dump(proj, f)
    print(f"Guardat: {name} | {len(images_out)} imatges | {sum(len(i['bayas']) for i in images_out)} baies")
print('Funcions carregades ✔')

In [ ]:
# 2) Detector de safata DEFINITIU: canal a* (verd) de LAB + watershed
#     Detecta baies verdes (clares I fosques), ignora paper/regle/cinta/text. Sense SAM 3.
import cv2, numpy as np

CALIBRA_1     = True       # True = només PREVIEW_IDX | False = totes
PREVIEW_IDX   = 0
MIN_AREA_FRAC = 0.00003    # àrea mínima de baia
MAX_AREA_FRAC = 0.004      # àrea màxima (treu cinta/grumolls grossos)
ASPECT_MAX    = 2.5        # descarta formes allargades (tija/raquis)
SEED_FRAC     = 0.35       # llavor watershed
PEAK_K        = 0.5        # separació entre baies (x radi típic); baixa per grups densos

def _empty(path, img, W, H):
    return {'nombre': path, 'dataURL': data_url(img), 'w': W, 'h': H, 'escala_mm_px': None,
            'escalaLinea': None, 'bayas': [], 'racimo': None,
            'ficha': dict(EMPTY_FICHA), 'verdad': dict(EMPTY_VERDAD)}

def process_tray(path):
    img = open_img(path); W, H = img.size
    bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    a = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)[:, :, 1]      # canal verd-vermell
    thr, _ = cv2.threshold(a, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    berry = ((a < thr).astype(np.uint8)) * 255            # verd = valors baixos
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    berry = cv2.morphologyEx(berry, cv2.MORPH_OPEN, k, iterations=1)
    berry = cv2.morphologyEx(berry, cv2.MORPH_CLOSE, k, iterations=1)

    dist = cv2.distanceTransform(berry, cv2.DIST_L2, 5)
    if dist.max() < 2:
        return _empty(path, img, W, H)
    hi = dist[dist > 0.5 * dist.max()]
    r0 = max(4.0, float(np.median(hi)) if hi.size else float(dist.max()))
    ks = int(max(3, round(r0 * PEAK_K))); ks += (ks + 1) % 2
    dil = cv2.dilate(dist, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ks, ks)))
    peaks = ((dist >= dil - 1e-3) & (dist > SEED_FRAC * r0)).astype(np.uint8)
    npk, pl = cv2.connectedComponents(peaks)
    if npk <= 1:
        return _empty(path, img, W, H)
    markers = np.zeros(dist.shape, np.int32)
    markers[cv2.erode((berry == 0).astype(np.uint8), k, iterations=2) > 0] = 1
    for lid in range(1, npk):
        markers[pl == lid] = lid + 1
    markers = cv2.watershed(bgr, markers)

    amin = MIN_AREA_FRAC * W * H; amax = MAX_AREA_FRAC * W * H
    bayas = []
    for lid in range(2, int(markers.max()) + 1):
        ys, xs = np.where(markers == lid)
        ar = len(xs)
        if ar < amin or ar > amax:
            continue
        wbb = xs.max() - xs.min() + 1; hbb = ys.max() - ys.min() + 1
        if max(wbb, hbb) / max(1, min(wbb, hbb)) > ASPECT_MAX:
            continue
        bayas.append({'cx': round(float(xs.mean()), 2), 'cy': round(float(ys.mean()), 2),
                      'r': round(float(np.sqrt(ar / np.pi)), 2)})

    return {'nombre': path, 'dataURL': data_url(img), 'w': W, 'h': H, 'escala_mm_px': None,
            'escalaLinea': None, 'bayas': bayas, 'racimo': None,
            'ficha': dict(EMPTY_FICHA), 'verdad': dict(EMPTY_VERDAD)}

paths = [image_paths[PREVIEW_IDX]] if CALIBRA_1 else image_paths
images_out, fallades = [], []
for n, path in enumerate(paths, 1):
    try:
        im = process_tray(path); images_out.append(im)
        print(f"[{n}/{len(paths)}] {os.path.basename(path)}: {len(im['bayas'])} baies")
    except Exception as e:
        fallades.append(path); print(f"[{n}/{len(paths)}] {path}: ERROR ({e})")
if fallades: print('Fallades:', fallades)
save_project(images_out)


In [ ]:
# 3) Previsualitzar (comprova abans de baixar)
import matplotlib.pyplot as plt
im0 = images_out[0]; img = open_img(im0['nombre'])
fig, ax = plt.subplots(figsize=(9, 11)); ax.imshow(img)
for b in im0['bayas']:
    ax.add_patch(plt.Circle((b['cx'], b['cy']), b['r'], fill=False, color='lime', lw=1.3))
ax.set_title(f"{len(im0['bayas'])} baies"); ax.axis('off'); plt.show()

In [ ]:
# 4) (Opcional) Processar per LOTS de 20 i descarregar cada lot
BATCH_SIZE = 20
BATCH      = 1     # 1 = imatges 1-20, 2 = 21-40, ...
start = (BATCH-1)*BATCH_SIZE; chunk = image_paths[start:start+BATCH_SIZE]
print(f"Lot {BATCH}: {len(chunk)} imatges ({start+1}-{start+len(chunk)} de {len(image_paths)})")
images_out = []
for n, path in enumerate(chunk, 1):
    try:
        im = process_tray(path); images_out.append(im)
        print(f"  [{n}/{len(chunk)}] {os.path.basename(path)}: {len(im['bayas'])} baies")
    except Exception as e:
        print(f"  [{n}/{len(chunk)}] {path}: ERROR ({e})")
save_project(images_out, name=f'modelvinya_safata_lote{BATCH}.json')
from google.colab import files
files.download(f'modelvinya_safata_lote{BATCH}.json')

In [ ]:
# 5) Descarregar el projecte (si has fet servir la cel·la 2 amb CALIBRA_1=False)
from google.colab import files
files.download('modelvinya_safata.json')

## A l'anotador
1. **https://planessoria-ui.github.io/ModelVinya/** → **📦 Abrir proyecto** → tria el `.json`.
2. Revisa els cercles (afegeix/esborra/ajusta).
3. **📏 Escala** sobre el regle (p. ex. 0→10 cm = 100 mm) → diàmetres en **mm**.
4. Omple la **verdad terreno** (calibre real) si la tens, i exporta **CSV** / **YOLO**.

### Ajustos (canvia 1 valor i torna a executar la cel·la 2 → 3)
- Falten baies groguenques → baixa `HUE_MIN` a 25.
- Agafa paper/ombres → puja `SAT_MIN` a 50.
- Uneix baies que es toquen → baixa `SEED_FRAC` a 0.4 i `PEAK_K` a 1.0.
- Parteix una baia en dues → puja `PEAK_K` a 1.5.